####Introduction to Aggregation
 Aggregation means calculating kind of a summary. Aggregation in Spark are implemented using functions

#####Comonly used aggregate functions
 1. count(*), count(expr), count(DISTINCT expr)
 2. min(expr), max(expr), avg(expr), sum(expr)

#####Types of Aggregation
 1. Simple Aggregation
 2. Grouped Aggregation
 3. Multilevel Aggregation
 4. Window Aggregation

####Requirement - Analysis Data Set

 Prepare club bookings dataset for analysis
 ```
 +----------+------------+---------------+-------------------+--------------+
 |booking_id| member_name|  facility_name|         start_time|booking_amount|
 +----------+------------+---------------+-------------------+--------------+
 ```

In [0]:
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

club_bookings_df = (
    bookings_df.join(facilities_df, "facility_id")
            .join(members_df, "member_id", "left")
            .selectExpr("booking_id",
                        "case when member_id==0 then 'Guest Member' else concat_ws(' ', first_name, last_name) end as member_name",
                        "facility_name","start_time",
                        "case when member_id == 0 then slots * guest_cost else slots * member_cost end as booking_amount")
)
club_bookings_df.display()

####1. Calculate total earnings and average booking value.

1.1 Using sql like expressions

In [0]:
result_df = (
    club_bookings_df.selectExpr("sum(booking_amount) as total_earning",
                                "avg(booking_amount) as avg_booking_value")
)

result_df.display()

1.2 Using column expressions

In [0]:
from pyspark.sql.functions import sum, avg
result_df = (
    club_bookings_df.select(sum("booking_amount").alias("total_earning"),
                                avg("booking_amount").alias("avg_booking_value")
    )
)
result_df.display()

1.3 Using aggregate transformation

In [0]:
from pyspark.sql.functions import sum, avg

result_df = (
    club_bookings_df.agg(sum("booking_amount").alias("total_earning"),
                         avg("booking_amount").alias("avg_booking_value"))
)

result_df.display()

####Grouping Aggregation
----------------------------------------------------------------------------------------------------------------------------------------------

####Requirement - Analysis Data Set

 Prepare club bookings dataset for analysis
 ```
 +----------+------------+---------------+-------------------+--------------+
 |booking_id| member_name|  facility_name|         start_time|booking_amount|
 +----------+------------+---------------+-------------------+--------------+
 ```

Q1. Who are the top 5 members by total booking amount?

 Prepare a report as the following.
 ```
 member_name     | total_booking_amount
 ---------------------------------------
 Tim Rownam      | 6480
 Tim Boothe      | 3644
 Gerald Butters  | 3343
 Burton Tracy    | 2953
 David Jones     | 2651
 ```

1.1 Try aggregation using agg() transformation

In [0]:
from pyspark.sql.functions import expr, col
result_df = (
    club_bookings_df.where("member_name !='Guest Member'")
    .groupBy("member_name")
    .agg( expr("sum(booking_amount)as total_booking_amount"))
    .orderBy(col("total_booking_amount").desc())
    .limit(5)
)

result_df.display()

Q2. Who are the members having total booking amount > 2500?

In [0]:
from pyspark.sql.functions import expr, col
result_df = (
    club_bookings_df.where("member_name !='Guest Member'")
    .groupBy("member_name")
    .agg( expr("sum(booking_amount)as total_booking_amount"))
    .filter(col("total_booking_amount") > 2500)
)

result_df.display()

Q3. Find member wise facility bookings for more than 2500?
```
 +-----------+--------------+--------------------+
 |member_name| facility_name|total_booking_amount|
 +-----------+--------------+--------------------+
 | Tim Boothe|Massage Room 1|              2660.0|
 | Tim Rownam|Massage Room 1|              6160.0|
 +-----------+--------------+--------------------+
 ```

In [0]:
from pyspark.sql.functions import expr, col

result_df = (
    club_bookings_df.where("member_name != 'Guest Member'")
            .groupBy("member_name", "facility_name")
            .agg(expr("sum(booking_amount) as total_booking_amount"))
            .where("total_booking_amount > 2500")
)

result_df.display()

###Multilevel Aggregation
----------------------------------------------------------------------------------------------------------------------------------------------

####Multilevel Aggregates
 Multilevel aggregates allows summarizing data at several hierarchical levels.\
 We have three variations of multilevel aggregates in Spark.
 1. Rollup
 2. Cube
 3. Grouping Sets

####Requirement - Analysis Data Set

 Prepare club bookings dataset for analysis
 ```
 +----------+------------+---------------+-------------------+--------------+
 |booking_id| member_name|  facility_name|         start_time|booking_amount|
 +----------+------------+---------------+-------------------+--------------+
 ```

In [0]:
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

club_bookings_df = (
    bookings_df.join(facilities_df, "facility_id")
            .join(members_df, "member_id", "left")
            .selectExpr("member_id", "booking_id",
                        "case when member_id==0 then 'Guest Member' else concat_ws(' ', first_name, last_name) end as member_name",
                        "facility_name","start_time",
                        "case when member_id == 0 then slots * guest_cost else slots * member_cost end as booking_amount")
)

club_bookings_df.display()

Q1. Prepare a monthly revenue report for year 2022.

 Also roll up the total for the month column.
 ```
 +----+--------+
 |mnth| revenue|
 +----+--------+
 |   7| 23202.5|
 |   8| 46066.5|
 |   9| 63315.5|
 |    |132584.5|
 +----+--------+
 ```

In [0]:
from pyspark.sql.functions import sum, col, month

result_df = (
    club_bookings_df.where("year(start_time)==2022")
    .withColumn("mnth", month("start_time"))
    .rollup("mnth") #rollup is a specialized group by which will do group by month and calculate the aggregate for each group and also rollup for that group at the end
    .agg(sum("booking_amount").alias("revenue"))
    .orderBy(col("mnth").asc_nulls_last())
)
result_df.display()

Q2. Prepare a revenue report by revenue_from (Guest/Member) and facility_name for year 2022.

 Also roll up the total for each group.
 ```
 +------------+---------------+-------+
 |revenue_from|  facility_name|revenue|
 +------------+---------------+-------+
 |       Guest|Badminton Court| 1906.5|
 |       Guest| Massage Room 1|41600.0|
 |       Guest| ..............|.......|
 |       Guest|     ..........|  .....|
 |       Guest|           NULL|89096.5|
 |      Member|Badminton Court|    0.0|
 |      Member| Massage Room 1|30940.0|
 |      Member| ..............| ......|
 |      Member| ..............| ......|
 |      Member|           NULL|43488.0|
 +------------+---------------+-------+
 ```

In [0]:
from pyspark.sql.functions import sum, col, expr

result_df = (
    club_bookings_df.where("year(start_time)==2022")
    .withColumn("revenue_from", expr("case when member_id==0 then 'Guest' else 'Member' end"))
    .rollup("revenue_from","facility_name")
    .agg(sum("booking_amount").alias("revenue"))
    .orderBy(col("revenue_from").asc_nulls_last(), col("facility_name").asc_nulls_last())

)
result_df.display()

Q3. Prepare a revenue report by revenue_from(Guest/Member) and facility_name for year 2022.

 Also compute totals for all 4 dimensions of revenue_from and facility_name.
 * (revenue_from, facility_name) : rollup
 * (revenue_from, ) : rollup
 * (facility_name, ) : not available in roolup
 * ( , ) : grand total in rollup

In [0]:
from pyspark.sql.functions import sum, col, expr

result_df = (
    club_bookings_df.where("year(start_time)==2022")
    .withColumn("revenue_from", expr("case when member_id==0 then 'Guest' else 'Member' end"))
    .cube("revenue_from","facility_name") #all possible aggregation combinations
    .agg(sum("booking_amount").alias("revenue"))
    .orderBy(col("revenue_from").asc_nulls_last(), col("facility_name").asc_nulls_last())

)
result_df.display()

Q4: Prepare a revenue report similar to the following.
 ```
   revenue_from  | facility_name   | revenue
   ---------------------------------------
   Guest         | Badminton Court | 1906.5
   Guest         | Massage Room 1  | 41600
   Guest         | Massage Room 2  | 13920
   Guest         |                 | 57426.5
   Member        | Badminton Court | 0
   Member        | Massage Room 1  | 30940
   Member        | Massage Room 2  | 1890
   Member        |                 | 32830
 ```
 Roll up the total for a the facility_name only.


In [0]:
from pyspark.sql.functions import sum, col, expr

result_df = (
    club_bookings_df.where("year(start_time)==2022")
    .withColumn("revenue_from", expr("case when member_id==0 then 'Guest' else 'Member' end"))
    .groupingSets([("revenue_from","facility_name"), ("revenue_from", )], "revenue_from","facility_name") #GROUPING SETS lets you define exact group combinations manually (instead of automatic ones like rollup/cube).
    .agg(sum("booking_amount").alias("revenue"))
    .orderBy(col("revenue_from").asc_nulls_last(), col("facility_name").asc_nulls_last())

)
result_df.display()

###Windowing Aggregation
----------------------------------------------------------------------------------------------------------------------------------------------

#####Window Aggregation
 Window aggregation performs calculations (like SUM, AVG, MIN, MAX) on a set of related rows (a "window").

#####Use Cases
 1. Running/Moving Aggregates
 2. Grouped top N
 3. Forward/Backward comparison

#####Structure

 ```
   
   agg_function().OVER(Window.PARTITION_BY(column_list)
                             .ORDER_BY(column_list)
                             .ROWS_BETWEEN(window_start, window_end)
                             
 ```

Q1. Prepare a daily revenue report for club facility bookings as shown below.\
 Add a running total to your report.
 ```
   booked_by | booking_date    | revenue | running_total
   -------------------------------------------------------
   Guest     | 2022-07-03      | 35      | 35
   Guest     | 2022-07-04      | 390     | 425
   Guest     | 2022-07-05      | 110     | 535
   Guest     | 2022-07-06      | 150     | 685
   Member    | 2022-07-03      | 70      | 70
   Member    | 2022-07-04      | 107     | 177
   Member    | 2022-07-05      | 77      | 254
   Member    | 2022-07-06      | 92      | 346

 ```

1.1 Prepare a daily revenue report.

In [0]:
from pyspark.sql.functions import expr, sum

bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

booking_summary_df = (
    bookings_df.join(facilities_df, "facility_id")
            .join(members_df, "member_id", "left")
            .where("month(start_time) = 7 AND year(start_time) = 2022")
            .withColumns({
                "booked_by": expr("case when member_id==0 then 'Guest' else 'Member' end"),
                "booking_date": expr("to_date(start_time)"),
                "booking_amount": expr("case when member_id == 0 then slots * guest_cost else slots * member_cost end")
                })
            .groupBy("booked_by", "booking_date")
            .agg(sum("booking_amount").alias("revenue"))
            .orderBy("booking_date")
)

booking_summary_df.display()

1.2 Add running total to your report.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum

window_spec = (
    Window.partitionBy("booked_by")
    .orderBy("booking_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)#UNBOUNDED PRECEDING defines a window starting from the first row of the partition up to the current row
)

result_df = (
    booking_summary_df.withColumn("running_total", sum("revenue").over(window_spec))
)

result_df.display()




Q2. Add a 3 day moving average to your revenue report
 ```
   booked_by   | booking_date    |revenue  | 3_day_avg
   -------------------------------------------------------
   Guest       | 2022-07-03      | 35      | 35
   Guest       | 2022-07-04      | 390     | 212.5
   Gues        | 2022-07-05      | 110     | 178.33
   Guest       | 2022-07-06      | 150     | 216.67
   Guest       | 2022-07-07      | 305     | 188.33
   Guest       | 2022-07-08      | 550     | 335
   Member      | 2022-07-03      | 70      | 70
   Member      | 2022-07-04      | 107     | 88.5
   Member      | 2022-07-05      | 77      | 84.67
   Member      | 2022-07-06      | 92      | 92
   Member      | 2022-07-07      | 199     | 122.67
 ```

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg, round

window_spec = (
    Window.partitionBy("booked_by")
    .orderBy("booking_date")
    .rowsBetween(-2,Window.currentRow)
)

result_df = (
    booking_summary_df.withColumn("3_day_avg",
                                  round(avg("revenue").over(window_spec),2))
)

result_df.display()

Q3. Show top 3 revenue dates from guests and members.

 Expected Results
 ```
   booked_by | booking_date  |revenue
   ----------------------------------
   Guest     | 2022-07-24    |1105
   Guest     | 2022-07-27    |990
   Guest     | 2022-07-30    |986.5
   Member    | 2022-07-25    |626
   Member    | 2022-07-31    |486
   Member    | 2022-07-26    |455
 ```

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, rank #assigns rank after order by

window_spec = (
    Window.partitionBy("booked_by")
    .orderBy(col("revenue").desc())
)

result_df = (
    booking_summary_df.withColumn("rank", rank().over(window_spec))
    .where(col("rank")<=3)
    .drop("rank")
)

result_df.display()